# Cybersecurity Intrusion Detection with XAI and LLMs

Organized notebook for GitHub and academic reproducibility.

## Install Dependencies

In [ ]:
!pip install shap lime -q

## Imports

In [ ]:
# Standard library
import json

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Explainability
import shap
from lime import lime_tabular

# OpenAI API
from openai import OpenAI

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Model selection
from sklearn.model_selection import train_test_split

# Machine Learning
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


## Configuration

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.30
SHAP_SAMPLE_SIZE = 200

DATASET_PATH = (
    "/content/sample_data/"
    "cybersecurity_intrusion_data.csv"
)


## Dataset Loading

In [ ]:
df = pd.read_csv(DATASET_PATH)

print("Dataset Shape:", df.shape)

df.head()


## Data Cleaning

In [ ]:
print(df.isnull().sum())

df.drop(columns=["session_id"], inplace=True)

df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

print("Cleaned dataset shape:", df.shape)


## Categorical Encoding

In [ ]:
categorical_columns = [
    "protocol_type",
    "encryption_used",
    "browser_type"
]

for column in categorical_columns:

    df[column] = (
        df[column]
        .astype("category")
    )

    print(
        f"{column} categories:",
        df[column].cat.categories.tolist()
    )

    df[column] = df[column].cat.codes


## Feature Scaling

In [ ]:
scaler = StandardScaler()

df[[
    "network_packet_size",
    "session_duration"
]] = scaler.fit_transform(
    df[[
        "network_packet_size",
        "session_duration"
    ]]
)


## Target Distribution

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    x="attack_detected",
    hue="attack_detected",
    data=df,
    palette="viridis",
    legend=False
)

plt.title(
    "Target Distribution "
    "(0: Normal | 1: Attack)"
)

plt.xlabel("attack_detected")
plt.ylabel("Count")

plt.show()


## Train/Test Split

In [ ]:
X = df.drop(columns=["attack_detected"])
y = df["attack_detected"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)


## Random Forest Training

In [ ]:
rf_model = RandomForestClassifier(
    random_state=RANDOM_STATE
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

print(
    classification_report(
        y_test,
        y_pred
    )
)


## Confusion Matrix

In [ ]:
matrix = confusion_matrix(
    y_test,
    y_pred
)

matrix = (
    matrix.astype("float")
    / matrix.sum(axis=1)[:, np.newaxis]
)

class_names = [
    "Normal",
    "Attack"
]

plt.figure(figsize=(6, 4))

sns.heatmap(
    matrix,
    annot=True,
    cmap=plt.cm.Blues,
    linewidths=0.2
)

plt.xlabel("Prediction")
plt.ylabel("Real")

plt.title("Dataset 2 Confusion Matrix")

plt.show()


## SHAP Explainability

In [ ]:
sample_idx = np.random.choice(
    X_test.index,
    size=min(
        SHAP_SAMPLE_SIZE,
        len(X_test)
    ),
    replace=False
)

X_sample = X_test.loc[sample_idx]

explainer = shap.TreeExplainer(
    rf_model
)

shap_values = explainer.shap_values(
    X_sample
)

shap.summary_plot(
    [
        shap_values[:, :, 0],
        shap_values[:, :, 1]
    ],
    X_sample,
    plot_type="bar",
    class_names=[
        "Normal",
        "Attack"
    ],
    show=False
)


## LIME Explainability

In [ ]:
feature_names = [
    "network_packet_size",
    "protocol_type",
    "login_attempts",
    "session_duration",
    "encryption_used",
    "ip_reputation_score",
    "failed_logins",
    "browser_type",
    "unusual_time_access"
]

class_names = [
    "Normal",
    "Attack"
]

lime_explainer = lime_tabular.LimeTabularExplainer(
    X_test.values,
    feature_names=feature_names,
    class_names=class_names,
    mode="classification",
    verbose=True
)

instance_index = 50

instance_to_explain = (
    X_test.iloc[instance_index]
    .values
)

lime_explanation = (
    lime_explainer.explain_instance(
        instance_to_explain,
        rf_model.predict_proba
    )
)

print(lime_explanation.as_list())

lime_explanation.show_in_notebook(
    show_table=True
)


## LLM Context Information

In [ ]:
column_description = {

    "network_packet_size": (
        "Packet payload size"
    ),

    "protocol_type": (
        "Transport layer protocol"
    ),

    "login_attempts": (
        "Total login attempts"
    ),

    "session_duration": (
        "Session duration"
    ),

    "encryption_used": (
        "Encryption algorithm used"
    ),

    "ip_reputation_score": (
        "IP reputation score"
    ),

    "failed_logins": (
        "Number of failed logins"
    ),

    "browser_type": (
        "Browser type used"
    ),

    "unusual_time_access": (
        "Access at unusual time"
    ),

    "attack_detected": (
        "Normal or Attack"
    )
}

category_encoding = {

    "protocol_type": {
        "ICMP": 0,
        "TCP": 1,
        "UDP": 2
    },

    "encryption_used": {
        "AES": 0,
        "DES": 1
    },

    "browser_type": {
        "Chrome": 0,
        "Edge": 1,
        "Firefox": 2,
        "Safari": 3,
        "Unknown": 4
    }
}

train_sample = X_train.sample(50)

train_sample["attack_detected"] = (
    y_train.loc[train_sample.index]
)

train_sample_json = train_sample.to_json(
    orient="records"
)

pred_sample = pd.DataFrame({
    "real": y_test,
    "predicted": y_pred
})

pred_sample_json = pred_sample.sample(
    50
).to_json(
    orient="records"
)

model_info = {
    "model_type": "Random Forest",
    "task": (
        "Intrusion detection using logs"
    ),
    "target_variable": "attack_detected",
    "features": list(X.columns)
}


## Prompt Construction

In [ ]:
prompt = f"""
You are an expert in Explainable Artificial
Intelligence (XAI) and Cybersecurity.

=========================
MODEL INFORMATION
=========================
{model_info}

=========================
COLUMN DESCRIPTION
=========================
{column_description}

=========================
CATEGORY ENCODING
=========================
{category_encoding}

=========================
TRAINING DATA SAMPLE
=========================
{train_sample_json}

=========================
REAL vs PREDICTED SAMPLE
=========================
{pred_sample_json}

=========================
TASK
=========================

1. Identify the top-3 most relevant features.

2. Compare differences between classes.

3. Interpret model behavior in cybersecurity.

4. Avoid causal claims.

The explanation should be understandable
for technical and non-technical users.
"""


## OpenAI API Call

In [ ]:
client = OpenAI(
    api_key="YOUR_API_KEY"
)

response = client.responses.create(
    model="gpt-5",
    input=[
        {
            "role": "system",
            "content": (
                "You are an expert in "
                "Machine Learning and XAI."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    max_output_tokens=12288
)


## Extract LLM Explanation

In [ ]:
explanation = ""

for item in response.output:

    if item.type == "message":

        for content in item.content:

            if content.type == "output_text":
                explanation += content.text

print(explanation)
